### Install Required Dependencies

In [2]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import umap
import plotly.express as px

### Download Pretrained Model from Hugging Face for Text Embeddings

In [7]:
# Load pretrained model
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
# Read in Awards DataFrame
df_awards = pd.read_csv('~/Desktop/ADA/ADA-final-project/data_ingestion/df_all_award_results_v3.csv').drop_duplicates()

In [9]:
# Missing descriptions by filling them with an empty string
df_awards['Description'] = df_awards['Description'].fillna('')

# Encode the descriptions
# show_progress_bar=True helps you track the 300k rows.
print("Starting encoding...")
embeddings = model.encode(df_awards['Description'].tolist(), 
                           batch_size=64, 
                           show_progress_bar=True)

# Embeddings is a 2D numpy array (300000, 384)
print(f"Embedding shape: {embeddings.shape}")

Starting encoding...


Batches:   0%|          | 0/4863 [00:00<?, ?it/s]

Embedding shape: (311182, 384)


### Save Embeddings to CSV

In [10]:
emb_cols = [f'v_{i}' for i in range(embeddings.shape[1])]
embedding_df = pd.DataFrame(embeddings, columns=emb_cols)

# Concatenate ID and Vectors
final_output = pd.concat([df_awards[['generated_internal_id']], embedding_df], axis=1)

# Save to CSV
final_output.to_csv('contract_embeddings.csv', index=False)

## Start In Cells below (embeddings already created)

In [11]:
# Read in Embeddings from CSV
final_output = pd.read_csv('contract_embeddings.csv')

### Explore Data

In [ ]:
# Take a representative sample - 30,000 for a clear visual
df_sample = df_awards.sample(100, random_state=42)
sample_embeddings = embeddings[df_awards.index]

# Initialize and run UMAP
# n_neighbors: higher = global view, lower = local detail
# min_dist: how tightly packed clusters should be
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, metric='cosine', random_state=42)
projections = reducer.fit_transform(sample_embeddings)

# Create a viz DataFrame
viz_df = pd.DataFrame(projections, columns=['x', 'y'])
viz_df['Description'] = df_sample['Description'].values
viz_df['Agency'] = df_sample['Awarding Agency'].values # Or whatever metadata you have

# 4. Plot with Plotly (Interactive)
fig = px.scatter(viz_df, x='x', y='y', 
                 hover_data=['Description'], 
                 color='Agency', # Color by a category to see if it clusters
                 title='Federal Contract Embedding Map',
                 opacity=0.5)

fig.show()